
# Project 1 — fastText + MLP

## Architecture 1: Word Embeddings + Classifier

This notebook implements the first required architecture
for abductive commonsense reasoning on the ART dataset.

Architecture:
- Frozen pretrained fastText embeddings (300d)
- Mean pooling for each of the four text segments
- Concatenation (1200d)
- Linear (1200 → 128)
- ReLU
- Linear (128 → 2)

Only the classifier is trained.


In [1]:
import sys
from pathlib import Path

import torch
import torch.nn as nn

# Add repository root to Python path
repo_root = Path.cwd().parents[1]

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Load saved preprocessing artifacts
artifact_path = repo_root / "project1" / "artifacts" / "fasttext_art.pt"

artifacts = torch.load(
    artifact_path,
    map_location="cpu",
    weights_only=True
)

vocab = artifacts["vocab"]
embedding_weights = artifacts["embedding_weights"]

# Restore frozen embedding layer
embedding_layer = nn.Embedding.from_pretrained(
    embedding_weights,
    freeze=True,
    padding_idx=vocab["[PAD]"]
)

print("Vocabulary size:", len(vocab))
print("Embedding shape:", embedding_layer.weight.shape)
print("Embeddings frozen:", not embedding_layer.weight.requires_grad)

# Verify that the updated preprocessing artifacts are loaded
assert len(vocab) == 15038, (
    f"Expected updated vocabulary (15038), got {len(vocab)}"
)

assert tuple(embedding_weights.shape) == (15038, 300)
assert not embedding_layer.weight.requires_grad
assert torch.all(embedding_weights[vocab["[PAD]"]] == 0)

print("\nUpdated preprocessing artifacts: PASS")


Vocabulary size: 15038
Embedding shape: torch.Size([15038, 300])
Embeddings frozen: True

Updated preprocessing artifacts: PASS


In [2]:
from datasets import load_dataset
from project1.src.data_loader import create_dataloader

# Load ART dataset
dataset = load_dataset("amarfurt/art-nlp-hs26")

# Create DataLoaders
batch_size = 32

train_loader = create_dataloader(
    dataset["train"],
    vocab,
    batch_size=batch_size,
    shuffle=True
)

val_loader = create_dataloader(
    dataset["validation"],
    vocab,
    batch_size=batch_size,
    shuffle=False
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))


C:\Users\kimbo\anaconda3\envs\nlp-art-hs26\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Training batches: 526
Validation batches: 32


In [3]:
# Get one batch
batch = next(iter(train_loader))

input_ids = batch["input_ids"]
lengths = batch["lengths"]
labels = batch["labels"]

# Pass through frozen embedding layer
embeddings = embedding_layer(input_ids)

print("Input IDs:", input_ids.shape)
print("Embeddings:", embeddings.shape)
print("Labels:", labels.shape)

# Verify padding
print("\nSequence lengths correct:",
      torch.equal(
          lengths,
          (input_ids != vocab["[PAD]"]).sum(dim=1)
      ))

# Verify that each example has exactly 3 separators
sep_counts = (input_ids == vocab["[SEP]"]).sum(dim=1)

print("Each example has 3 separators:",
      torch.all(sep_counts == 3).item())


Input IDs: torch.Size([32, 58])
Embeddings: torch.Size([32, 58, 300])
Labels: torch.Size([32])

Sequence lengths correct: True
Each example has 3 separators: True


In [4]:

from project1.src.models import EmbeddingMLP

# Initialize model
model = EmbeddingMLP(
    embedding_layer=embedding_layer,
    pad_id=vocab["[PAD]"],
    sep_id=vocab["[SEP]"],
    hidden_dim=128
)

# Use our existing batch
batch = next(iter(train_loader))

# Forward pass
with torch.no_grad():
    logits = model(batch["input_ids"])

print("Input shape:", batch["input_ids"].shape)
print("Output shape:", logits.shape)

print("\nFirst 5 logits:")
print(logits[:5])

print("\nPredicted classes:")
print(logits.argmax(dim=1)[:5])

print("\nActual classes:")
print(batch["labels"][:5])

# Check trainable parameters
trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("\nTrainable parameters:", trainable_params)
print("Embeddings frozen:", not model.embedding.weight.requires_grad)


Input shape: torch.Size([32, 56])
Output shape: torch.Size([32, 2])

First 5 logits:
tensor([[-0.0173,  0.0533],
        [-0.0157,  0.0581],
        [-0.0120,  0.0595],
        [-0.0192,  0.0563],
        [-0.0151,  0.0612]])

Predicted classes:
tensor([1, 1, 1, 1, 1])

Actual classes:
tensor([0, 1, 1, 0, 1])

Trainable parameters: 153986
Embeddings frozen: True


In [5]:
import wandb

# Initialize a test experiment
run = wandb.init(
    project="nlp-art-hs26-project1",
    name="setup-test",
    job_type="setup",
    config={
        "model": "fasttext-mlp",
        "embedding_dim": 300,
        "hidden_dim": 128,
        "batch_size": 32
    }
)

# Log one test metric
run.log({"test/connection": 1})

print("Project:", run.project)
print("Entity:", run.entity)
print("Run URL:", run.url)

# Finish the experiment
run.finish()


wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from C:\Users\kimbo\_netrc.
wandb: Currently logged in as: kimbortis (kimbortis-hochschule-luzern) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


Project: nlp-art-hs26-project1
Entity: kimbortis-hochschule-luzern
Run URL: https://forge.coreweave.com/wandb/kimbortis-hochschule-luzern/nlp-art-hs26-project1/runs/yl357st7


test/connection,▁
test/connection,1


In [5]:
import torch.nn as nn
from project1.src.training import evaluate

# Our loss function
criterion = nn.CrossEntropyLoss()

# Use CPU for now
device = torch.device("cpu")
model = model.to(device)

# Evaluate the untrained model
val_metrics = evaluate(
    model,
    val_loader,
    criterion,
    device
)

print(f"Validation loss: {val_metrics['loss']:.4f}")
print(f"Validation accuracy: {val_metrics['accuracy']:.2%}")


Validation loss: 0.6944
Validation accuracy: 49.20%


In [6]:
import random
import numpy as np
import torch
import torch.nn as nn

from project1.src.models import EmbeddingMLP
from project1.src.training import train_one_epoch, evaluate

# Reproducibility
seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("Random seed:", seed)


# Hyperparameters
config = {
    "architecture": "fasttext-mlp",
    "embedding_dim": 300,
    "hidden_dim": 128,
    "batch_size": 32,
    "learning_rate": 0.001,
    "optimizer": "Adam",
    "max_epochs": 200,
    "patience": 4,
    "early_stopping": False,
    "seed": seed
}

# Dynamic device selection
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Selected device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# Initialize a new model
model = EmbeddingMLP(
    embedding_layer=embedding_layer,
    pad_id=vocab["[PAD]"],
    sep_id=vocab["[SEP]"],
    hidden_dim=config["hidden_dim"]
).to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=config["learning_rate"]
)

print("Device:", device)
print("Optimizer:", optimizer.__class__.__name__)
print("Learning rate:", config["learning_rate"])
print("Training examples:", len(train_loader.dataset))
print("Validation examples:", len(val_loader.dataset))


Random seed: 42
Selected device: cpu
Device: cpu
Optimizer: Adam
Learning rate: 0.001
Training examples: 16801
Validation examples: 1000


In [7]:
import wandb
from pathlib import Path

# Directory for checkpoints
checkpoint_dir = repo_root / "project1" / "artifacts"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# Descriptive experiment name
run_name = (
    f"mlp_h{config['hidden_dim']}"
    f"_lr{config['learning_rate']}"
    f"_bs{config['batch_size']}"
    f"_seed{config['seed']}"
    "_preprocV2_200ep_noES"
)

# Initialize W&B
run = wandb.init(
    entity="kimbortis-hochschule-luzern",
    project="nlp-art-hs26-project1",
    group="fasttext-mlp",
    job_type="train",
    name=run_name,
    config={
        **config,
        "preprocessing": "v2",
        "vocab_size": len(vocab),
        "embedding_frozen": True
    }
)

# Plot all metrics against the epoch
run.define_metric("epoch")
run.define_metric("train/*", step_metric="epoch")
run.define_metric("val/*", step_metric="epoch")

# Unique checkpoint for this experiment
checkpoint_path = checkpoint_dir / (
    f"{run_name}_{run.id}_best.pt"
)

best_val_accuracy = -1.0
best_val_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0

lowest_val_loss = float("inf")
lowest_loss_epoch = 0

print("W&B run:", run.name)
print("W&B URL:", run.url)
print("Checkpoint:", checkpoint_path)

try:
    for epoch in range(1, config["max_epochs"] + 1):

        # Training
        train_metrics = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            device
        )

        # Validation
        val_metrics = evaluate(
            model,
            val_loader,
            criterion,
            device
        )

        # Log metrics against epoch
        run.log({
            "epoch": epoch,
            "train/loss": train_metrics["loss"],
            "train/accuracy": train_metrics["accuracy"],
            "val/loss": val_metrics["loss"],
            "val/accuracy": val_metrics["accuracy"]
        })

        # Track lowest validation loss independently
        if val_metrics["loss"] < lowest_val_loss:
            lowest_val_loss = val_metrics["loss"]
            lowest_loss_epoch = epoch

            run.summary["lowest_val_loss"] = lowest_val_loss
            run.summary["lowest_loss_epoch"] = lowest_loss_epoch

        print(
            f"Epoch {epoch:02d} | "
            f"Train loss: {train_metrics['loss']:.4f} | "
            f"Train acc: {train_metrics['accuracy']:.2%} | "
            f"Val loss: {val_metrics['loss']:.4f} | "
            f"Val acc: {val_metrics['accuracy']:.2%}"
        )

        # Select by validation accuracy
        # Break ties using validation loss
        improved = (
            val_metrics["accuracy"] > best_val_accuracy
            or (
                val_metrics["accuracy"] == best_val_accuracy
                and val_metrics["loss"] < best_val_loss - 1e-4
            )
        )

        if improved:
            best_val_accuracy = val_metrics["accuracy"]
            best_val_loss = val_metrics["loss"]
            best_epoch = epoch
            epochs_without_improvement = 0

            # Save model parameters
            torch.save(
                model.state_dict(),
                checkpoint_path
            )

            # Store best results in W&B summary
            run.summary["best_epoch"] = best_epoch
            run.summary["best_val_accuracy"] = best_val_accuracy
            run.summary["val_loss_at_best_accuracy"] = best_val_loss

        else:
            epochs_without_improvement += 1

        # Early stopping (disabled for diagnostic runs)
        if (
            config["early_stopping"]
            and epochs_without_improvement >= config["patience"]
        ):
            print(f"\nEarly stopping at epoch {epoch}")
            break

    print("\n=== BEST MODEL ===")
    print("Best epoch:", best_epoch)
    print(f"Best validation accuracy: {best_val_accuracy:.2%}")
    print(f"Validation loss: {best_val_loss:.4f}")
    print("\n=== LOWEST VALIDATION LOSS ===")
    print("Epoch:", lowest_loss_epoch)
    print(f"Lowest validation loss: {lowest_val_loss:.4f}")
    print("Checkpoint:", checkpoint_path)

finally:
    run.finish()

wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from C:\Users\kimbo\_netrc.
wandb: Currently logged in as: kimbortis (kimbortis-hochschule-luzern) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


W&B run: mlp_h128_lr0.001_bs32_seed42_preprocV2_200ep_noES
W&B URL: https://forge.coreweave.com/wandb/kimbortis-hochschule-luzern/nlp-art-hs26-project1/runs/gyjy8tcu
Checkpoint: C:\Users\kimbo\Documents\HSLU\Sem 5\NLP\nlp-art-hs26-kim\project1\artifacts\mlp_h128_lr0.001_bs32_seed42_preprocV2_200ep_noES_gyjy8tcu_best.pt
Epoch 01 | Train loss: 0.6920 | Train acc: 51.93% | Val loss: 0.6889 | Val acc: 52.90%
Epoch 02 | Train loss: 0.6773 | Train acc: 57.39% | Val loss: 0.6746 | Val acc: 56.30%
Epoch 03 | Train loss: 0.6579 | Train acc: 61.16% | Val loss: 0.6649 | Val acc: 58.70%
Epoch 04 | Train loss: 0.6446 | Train acc: 62.57% | Val loss: 0.6553 | Val acc: 60.10%
Epoch 05 | Train loss: 0.6336 | Train acc: 63.75% | Val loss: 0.6589 | Val acc: 59.50%
Epoch 06 | Train loss: 0.6251 | Train acc: 65.17% | Val loss: 0.6496 | Val acc: 60.50%
Epoch 07 | Train loss: 0.6172 | Train acc: 65.53% | Val loss: 0.6495 | Val acc: 60.10%
Epoch 08 | Train loss: 0.6101 | Train acc: 66.56% | Val loss: 0.6508 |

epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇█
train/accuracy,▁▁▂▂▃▄▄▅▅▅▆▆▆▇▇▇████████████████████████
train/loss,█▇▇▇▆▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/accuracy,▁█▇█▄▇▇▇▅▆▄▅▆▅▆▄▅▆▆▆▅▄▆▅▆▅▆▅▅▆▅▅▅▄▅▄▅▄▄▄
val/loss,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▂▂▂▃▃▂▄▄▅▅▅▅▅▆▆▆▆▇▇▇█
best_epoch,47
best_val_accuracy,0.627
epoch,200
lowest_loss_epoch,14
lowest_val_loss,0.64392
train/accuracy,1


In [8]:
# Restore the best model checkpoint
model.load_state_dict(
    torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=True
    )
)

model.eval()

# Evaluate the restored model
best_metrics = evaluate(
    model,
    val_loader,
    criterion,
    device
)

print("=== RESTORED BEST MODEL ===")
print("Best epoch:", best_epoch)
print(f"Validation loss: {best_metrics['loss']:.4f}")
print(f"Validation accuracy: {best_metrics['accuracy']:.2%}")

# Verify checkpoint matches the recorded best result
assert abs(
    best_metrics["accuracy"] - best_val_accuracy
) < 1e-6

print("Checkpoint verification: PASS")

# Compare against the previous baseline
previous_accuracy = 0.622

difference = (
    best_metrics["accuracy"] - previous_accuracy
) * 100

print("\n=== BASELINE COMPARISON ===")
print(f"Previous MLP: {previous_accuracy:.2%}")
print(f"Updated MLP: {best_metrics['accuracy']:.2%}")
print(f"Difference: {difference:+.2f} percentage points")


=== RESTORED BEST MODEL ===
Best epoch: 47
Validation loss: 0.7677
Validation accuracy: 62.70%
Checkpoint verification: PASS

=== BASELINE COMPARISON ===
Previous MLP: 62.20%
Updated MLP: 62.70%
Difference: +0.50 percentage points
